# Reducción de dimensionalidad: PCA y t-SNE

Implementaciones breves e independientes y una comparación final sobre los mismos datos.

## Preparación de datos
Usaremos el conjunto de dígitos de `sklearn` (64 características por imagen). Estandarizamos las variables antes de aplicar ambos métodos.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

digits = load_digits()
X = StandardScaler().fit_transform(digits.data)
y = digits.target
print(f"Observaciones: {X.shape[0]}, características: {X.shape[1]}")

## 1. PCA
PCA construye componentes lineales que capturan la mayor varianza posible. Comparamos dos cantidades de componentes y revisamos la varianza explicada.

In [ ]:
pca_2 = PCA(n_components=2)
X_pca_2 = pca_2.fit_transform(X)
pca_10 = PCA(n_components=10)
X_pca_10 = pca_10.fit_transform(X)

print(f"Varianza explicada con 2 componentes: {pca_2.explained_variance_ratio_.sum():.1%}")
print(f"Varianza explicada con 10 componentes: {pca_10.explained_variance_ratio_.sum():.1%}")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].scatter(X_pca_2[:, 0], X_pca_2[:, 1], c=y, cmap="tab10", s=8, alpha=0.7)
ax[0].set_title("PCA: 2 componentes")
ax[0].set_xlabel("PC1"); ax[0].set_ylabel("PC2")
ax[1].bar(range(1, 11), pca_10.explained_variance_ratio_)
ax[1].set_title("PCA: varianza explicada (10 componentes)")
ax[1].set_xlabel("Componente"); ax[1].set_ylabel("Proporción de varianza")
plt.tight_layout(); plt.show()

## 2. t-SNE
t-SNE busca preservar principalmente las relaciones entre vecinos cercanos. Comparamos dos valores de `perplexity`. Las coordenadas resultantes no representan componentes principales.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for eje, perplexity in zip(ax, [5, 40]):
    X_tsne = TSNE(n_components=2, perplexity=perplexity, random_state=42, init="pca", learning_rate="auto").fit_transform(X)
    eje.scatter(X_tsne[:, 0], X_tsne[:, 1], c=y, cmap="tab10", s=8, alpha=0.7)
    eje.set_title(f"t-SNE: perplexity={perplexity}")
    eje.set_xlabel("Dimensión 1"); eje.set_ylabel("Dimensión 2")
plt.tight_layout(); plt.show()

## 3. Comparación: PCA vs. t-SNE
Aplicamos ambos métodos a **los mismos datos** y mostramos sus proyecciones en 2D. Los colores corresponden al dígito real, no a clusters calculados.

In [ ]:
X_tsne = TSNE(n_components=2, perplexity=30, random_state=42, init="pca", learning_rate="auto").fit_transform(X)

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for eje, datos, titulo in zip(ax, [X_pca_2, X_tsne], ["PCA (2 componentes)", "t-SNE (perplexity=30)"]):
    sc = eje.scatter(datos[:, 0], datos[:, 1], c=y, cmap="tab10", s=9, alpha=0.7, vmin=-0.5, vmax=9.5)
    eje.set_title(titulo)
    eje.set_xlabel("Dimensión 1"); eje.set_ylabel("Dimensión 2")
fig.colorbar(sc, ax=ax, ticks=range(10), label="Dígito")
plt.show()

### Conclusiones

- **PCA:** permite cuantificar la varianza explicada y obtener una reducción lineal.
- **t-SNE:** suele mostrar mejor ciertas vecindades locales, pero su visualización depende de los parámetros.
- **Importante:** ninguno de los dos es un algoritmo de clustering; la separación visual no demuestra por sí sola que existan clusters. Las distancias globales en t-SNE deben interpretarse con cautela.